In [17]:
#Auto ML - Student Application
# Install Libraries
!pip install -q pandas numpy scikit-learn joblib

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
# Create a Meaningful Dataset
np.random.seed(42)

n = 500

data = pd.DataFrame({

    "Study_Hours": np.random.randint(1, 10, n),

    "Attendance": np.random.randint(50, 101, n),

    "Assignment_Score": np.random.randint(40, 101, n),

    "Internal_Marks": np.random.randint(35, 101, n),

    "Previous_Semester_Marks": np.random.randint(40, 101, n),

    "Sleep_Hours": np.random.uniform(4, 9, n).round(1),

    "Participation": np.random.randint(0, 11, n),

    "Backlogs": np.random.randint(0, 5, n)
})

# Create meaningful target
score = (
    data["Study_Hours"] * 5
    + data["Attendance"] * 0.25
    + data["Assignment_Score"] * 0.25
    + data["Internal_Marks"] * 0.30
    + data["Previous_Semester_Marks"] * 0.20
    + data["Sleep_Hours"] * 2
    + data["Participation"] * 2
    - data["Backlogs"] * 10
)

# Add some randomness
score = score + np.random.normal(0, 15, n)

data["Result"] = np.where(
    score >= np.median(score),
    1,
    0
)

print("Dataset created successfully!")
print("Shape:", data.shape)

data.head()

Dataset created successfully!
Shape: (500, 9)


,Study_Hours,Attendance,Assignment_Score,Internal_Marks,Previous_Semester_Marks,Sleep_Hours,Participation,Backlogs,Result
0,7,80,43,73,51,4.6,5,0,1
1,4,68,61,52,44,8.3,7,1,0
2,8,93,41,93,73,5.2,4,1,1
3,5,88,49,51,87,7.3,2,4,0
4,7,76,44,48,76,8.1,5,0,1


In [3]:
print("Dataset Information")
print("=" * 50)

print(data.info())

print("\nClass Distribution:")
print(data["Result"].value_counts())

print("\nClass Meaning:")
print("0 = Fail")
print("1 = Pass")

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Study_Hours              500 non-null    int32  
 1   Attendance               500 non-null    int32  
 2   Assignment_Score         500 non-null    int32  
 3   Internal_Marks           500 non-null    int32  
 4   Previous_Semester_Marks  500 non-null    int32  
 5   Sleep_Hours              500 non-null    float64
 6   Participation            500 non-null    int32  
 7   Backlogs                 500 non-null    int32  
 8   Result                   500 non-null    int64  
dtypes: float64(1), int32(7), int64(1)
memory usage: 21.6 KB
None

Class Distribution:
Result
1    250
0    250
Name: count, dtype: int64

Class Meaning:
0 = Fail
1 = Pass


In [4]:
X = data.drop("Result", axis=1)

y = data["Result"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print("Result")

Features:
['Study_Hours', 'Attendance', 'Assignment_Score', 'Internal_Marks', 'Previous_Semester_Marks', 'Sleep_Hours', 'Participation', 'Backlogs']

Target:
Result


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 375
Testing samples: 125


In [6]:
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

In [7]:
param_grids = {

    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10]
    },

    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": [
            "uniform",
            "distance"
        ]
    },

    "Decision Tree": {
        "model__max_depth": [
            2, 3, 4, 5, None
        ],
        "model__min_samples_split": [
            2, 5, 10
        ]
    },

    "Random Forest": {
        "model__n_estimators": [
            50, 100, 150
        ],
        "model__max_depth": [
            3, 5, 8, None
        ]
    },

    "SVM": {
        "model__C": [
            0.1, 1, 10
        ],
        "model__kernel": [
            "linear",
            "rbf"
        ]
    }
}

In [8]:
results = []

best_model = None
best_model_name = None
best_accuracy = 0
best_parameters = None

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print("Training:", model_name)
    print("=" * 70)

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    # Best tuned model
    tuned_model = grid.best_estimator_

    # Prediction
    y_pred = tuned_model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    recall = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    print("Best Parameters:")
    print(grid.best_params_)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Best Parameters": grid.best_params_
    })

    # Select best model
    if accuracy > best_accuracy:

        best_accuracy = accuracy
        best_model = tuned_model
        best_model_name = model_name
        best_parameters = grid.best_params_


Training: Logistic Regression
Best Parameters:
{'model__C': 0.1}
Accuracy : 0.8080
Precision: 0.7879
Recall   : 0.8387
F1 Score : 0.8125

Training: KNN
Best Parameters:
{'model__n_neighbors': 9, 'model__weights': 'uniform'}
Accuracy : 0.8000
Precision: 0.7761
Recall   : 0.8387
F1 Score : 0.8062

Training: Decision Tree
Best Parameters:
{'model__max_depth': None, 'model__min_samples_split': 2}
Accuracy : 0.6960
Precision: 0.6818
Recall   : 0.7258
F1 Score : 0.7031

Training: Random Forest
Best Parameters:
{'model__max_depth': 8, 'model__n_estimators': 150}
Accuracy : 0.7920
Precision: 0.7647
Recall   : 0.8387
F1 Score : 0.8000

Training: SVM
Best Parameters:
{'model__C': 10, 'model__kernel': 'linear'}
Accuracy : 0.8080
Precision: 0.7969
Recall   : 0.8226
F1 Score : 0.8095


In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("=" * 80)
print("AUTOML MODEL COMPARISON")
print("=" * 80)

display(results_df)

AUTOML MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Logistic Regression,0.808,0.787879,0.838710,0.812500,{'model__C': 0.1}
4,SVM,0.808,0.796875,0.822581,0.809524,"{'model__C': 10, 'model__kernel': 'linear'}"
1,KNN,0.800,0.776119,0.838710,0.806202,"{'model__n_neighbors': 9, 'model__weights': 'u..."
3,Random Forest,0.792,0.764706,0.838710,0.800000,"{'model__max_depth': 8, 'model__n_estimators':..."
2,Decision Tree,0.696,0.681818,0.725806,0.703125,"{'model__max_depth': None, 'model__min_samples..."


In [10]:
print("=" * 70)
print("BEST MODEL")
print("=" * 70)

print("Model:", best_model_name)

print(
    "Accuracy:",
    round(best_accuracy, 4)
)

print(
    "Best Parameters:",
    best_parameters
)

BEST MODEL
Model: Logistic Regression
Accuracy: 0.808
Best Parameters: {'model__C': 0.1}


In [11]:
best_predictions = best_model.predict(X_test)

print("Classification Report")
print("=" * 70)

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=[
            "Fail",
            "Pass"
        ]
    )
)

Classification Report
              precision    recall  f1-score   support

        Fail       0.83      0.78      0.80        63
        Pass       0.79      0.84      0.81        62

    accuracy                           0.81       125
   macro avg       0.81      0.81      0.81       125
weighted avg       0.81      0.81      0.81       125



In [12]:
cm = confusion_matrix(
    y_test,
    best_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[49 14]
 [10 52]]


In [13]:
joblib.dump(
    best_model,
    "best_student_performance_model.pkl"
)

print("Best model saved successfully!")
print(
    "File: best_student_performance_model.pkl"
)

Best model saved successfully!
File: best_student_performance_model.pkl


In [14]:
loaded_model = joblib.load(
    "best_student_performance_model.pkl"
)

new_student = pd.DataFrame({

    "Study_Hours": [7],

    "Attendance": [88],

    "Assignment_Score": [82],

    "Internal_Marks": [78],

    "Previous_Semester_Marks": [75],

    "Sleep_Hours": [7],

    "Participation": [8],

    "Backlogs": [0]
})

prediction = loaded_model.predict(
    new_student
)

if prediction[0] == 1:
    result = "PASS"
else:
    result = "FAIL"

print("Student Prediction:", result)

Student Prediction: PASS


In [15]:
oaded_model = joblib.load(
    "best_student_performance_model.pkl"
)

new_student = pd.DataFrame({

    "Study_Hours": [3],

    "Attendance": [20],

    "Assignment_Score": [15],

    "Internal_Marks": [56],

    "Previous_Semester_Marks": [35],

    "Sleep_Hours": [7],

    "Participation": [8],

    "Backlogs": [0]
})

prediction = loaded_model.predict(
    new_student
)

if prediction[0] == 1:
    result = "PASS"
else:
    result = "FAIL"

print("Student Prediction:", result)

Student Prediction: FAIL
